# FirstAI — train from scratch on Colab
Choose a T4 GPU under Runtime → Change runtime type if one is available. A free GPU is not guaranteed.

In [ ]:
import os
if os.path.isdir('/content/FirstAI'):
    !git -C /content/FirstAI pull --ff-only
else:
    !git clone https://github.com/llakshyaa-rgb/FirstAI.git /content/FirstAI
%cd /content/FirstAI
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU runtime active. GPU training requires Runtime → Change runtime type → T4 GPU, if available.')

In [ ]:
# Run the project's automated checks before training.
!python -m unittest discover -s tests -v

## Test 3 — full pipeline on the attached GPU
Requires CUDA and refuses CPU fallback. Mounts Drive, trains a tiny synthetic model with mixed precision, evaluates it, resumes it, reloads it for generation, reports peak VRAM and elapsed time, and retains checkpoints in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!python scripts/smoke_test.py --require-cuda --keep-artifacts --output-dir /content/drive/MyDrive/FirstAI/gpu-smoke

## Upload your corpus documents
Upload one or more UTF-8 `.txt` files that you have permission to use. The files go into the temporary Colab runtime, not into the public GitHub repository.

In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
raw_dir = Path('data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)
txt_files = [(name, data) for name, data in uploaded.items() if name.lower().endswith('.txt')]
if not txt_files:
    raise RuntimeError('Upload at least one .txt file')
for name, data in txt_files:
    (raw_dir / Path(name).name).write_bytes(data)
print(f'Uploaded {len(txt_files)} text file(s)')
!python prepare_corpus.py

In [ ]:
!python train.py --steps 1000 --eval-interval 100 --checkpoint-interval 250 --batch-size 16 --block-size 128 --layers 4 --heads 4 --width 128 --grad-accum-steps 2 --amp

## Generate a sample
The checkpoint is saved in the temporary Colab runtime. Download it before the runtime disconnects.

In [ ]:
!python generate.py --prompt "The " --tokens 300 --temperature 0.8 --top-k 40

In [ ]:
from google.colab import files
files.download('checkpoints/firstgpt.pt')
files.download('checkpoints/firstgpt.best.pt')